# Tomato Disease Classification - Final Project
**Author:** Laksh Gupta | **Reg No:** 2430010393


In [ ]:
# ==========================================
# 1. SETUP & UNZIP DATASET
# ==========================================
!unzip -q "tomato_data.zip.zip" -d /content/

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import copy
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# ==========================================
# 2. DATA PREPARATION & DATALOADERS
# ==========================================
data_dir = "/content/Tomato_Disease_Dataset"

# Data Augmentation for training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Standard transform for val/test
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(f"{data_dir}/train", transform=train_transform)
val_dataset = datasets.ImageFolder(f"{data_dir}/val", transform=test_transform)
test_dataset = datasets.ImageFolder(f"{data_dir}/test", transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = train_dataset.classes
print("Classes:", classes)
print("Train samples:", len(train_dataset))


In [ ]:
# ==========================================
# 3. CUSTOM REGULARIZED CNN MODEL (Baseline)
# ==========================================
class RegularizedTomatoCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super(RegularizedTomatoCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

regularized_model = RegularizedTomatoCNN().to(device)

def regularization_loss(model, l1_lambda=1e-5, l2_lambda=1e-4):
    l1_loss = torch.tensor(0.0, device=device)
    l2_loss = torch.tensor(0.0, device=device)
    for param in model.parameters():
        l1_loss += torch.sum(torch.abs(param))
        l2_loss += torch.sum(param ** 2)
    return (l1_lambda * l1_loss) + (l2_lambda * l2_loss)



In [ ]:
# ==========================================
# 4. HIGH ACCURACY PROPOSED MODEL (Fine-Tuned ResNet18)
# ==========================================
print("Setting up Fine-Tuned ResNet18 for 93% Accuracy...")

model_tl = models.resnet18(weights='DEFAULT')

# UNFREEZE ALL LAYERS
for param in model_tl.parameters():
    param.requires_grad = True

num_ftrs = model_tl.fc.in_features
model_tl.fc = nn.Linear(num_ftrs, 10)
model_tl = model_tl.to(device)

criterion_tl = nn.CrossEntropyLoss()
# Apply L2 Regularization
optimizer_tl = optim.Adam(model_tl.parameters(), lr=0.0001, weight_decay=1e-4)
# Add LR Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_tl, mode='min', factor=0.5, patience=2)



In [ ]:
# ==========================================
# 5. TRAINING LOOP FOR RESNET18
# ==========================================
num_epochs = 15
best_val_loss = float('inf')
best_model_weights = copy.deepcopy(model_tl.state_dict())
patience = 3
epochs_without_improvement = 0

train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []

for epoch in range(num_epochs):
    model_tl.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_tl.zero_grad()
        outputs = model_tl(images)
        loss = criterion_tl(outputs, labels)
        loss.backward()
        optimizer_tl.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_loss = running_loss / len(train_dataset)
    train_acc = 100 * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # Validation
    model_tl.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_tl(images)
            loss = criterion_tl(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    val_loss = val_loss / len(val_dataset)
    val_acc = 100 * correct / total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model_tl.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model_tl.load_state_dict(best_model_weights)
print("Training Complete. Best weights restored.")



In [ ]:
# ==========================================
# 6. TEST SET EVALUATION & CONFUSION MATRIX
# ==========================================
model_tl.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_tl(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Fine-Tuned ResNet18')
plt.show()

